In [1]:
import sys
import os
from dotenv import load_dotenv
import guidance
from guidance import system, user, assistant, gen
import math
# Setup local pywhyllm development environment in one line
from notebook_setup import setup_local_pywhyllm
project_root = setup_local_pywhyllm()

# Now import the SimpleModelSuggester
#TFM/pywhyllm/pywhyllm/suggesters --> crear variable 

# project_root = os.path.abspath("/home/moleropa/repositories/master/TFM/pywhyllm/pywhyllm/suggesters/")  # Navigate to TFM/pywhyllm/
# sys.path.insert(0, project_root)

from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester

from openai import OpenAI
from portkey_ai import createHeaders
from dotenv import load_dotenv
import time
import base64
from IPython.display import display, Image
from pydantic import BaseModel
import json
load_dotenv()

📋 Current sys.path before adding project root:
  0: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  1: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  3: 
  4: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages

🎯 Project root to add: /home/moleropa/repositories/master/TFM/pywhyllm
✅ Added local pywhyllm source to Python path: /home/moleropa/repositories/master/TFM/pywhyllm

📋 Updated sys.path after adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm ⭐
  1: /home/moleropa/miniforge3/envs/tfmenv/lib/python311.zip
  2: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11
  3: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/lib-dynload
  4: 
  5: /home/moleropa/miniforge3/envs/tfmenv/lib/python3.11/site-packages
📋 Current sys.path before adding project root:
  0: /home/moleropa/repositories/master/TFM/pywhyllm
  1: /home/moleropa/miniforge3/envs/tfmenv/lib/pyth

True

In [2]:
azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"

portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])
# azure_openai_client = OpenAI(base_url=us_base_url,
#             api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
#             default_headers=portkey_headers)


# Guidance con modelo OpenAI + base_url + headers
model = guidance.models.OpenAI(
    #"GPT-4o-2024-05-13",
    azure_model,
    api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
    base_url=us_base_url,
    default_headers=portkey_headers,
)

azure_model= "gpt-4o-mini" #"GPT-4o-2024-05-13" 
us_base_url = "https://us.aigw.galileo.roche.com/v1"
portkey_headers = createHeaders(config=os.environ["PORTKEY_AZURE_US_CONFIG"])

azure_openai_client = OpenAI(base_url=us_base_url,
            api_key=os.environ["PORTKEY_AZURE_US_API_KEY"],
            default_headers=portkey_headers)


## Extracting ONLY the Answer Token Probabilities

When dealing with reasoning + answer format, we only care about the probabilities of the final answer token (A, B, or C), not the reasoning tokens.

In [3]:
from pywhyllm.suggesters.simple_model_suggester import SimpleModelSuggester

# Recreate the modeler
modeler = SimpleModelSuggester(llm=model)

Try with Structured output with logprobs gives us all tokens and their logprobs, but we can filter to just the answer tokens.

In [4]:
# Test the flexible function with both log_probs=True and log_probs=False
from types import MethodType
import json

# we used this but no longer needed Add the flexible method to the modeler
# modeler.suggest_pairwise_relationship_with_logprobs_flexible = MethodType(
#     suggest_pairwise_relationship_with_logprobs_flexible, modeler
# )

result_with_logprobs = modeler.suggest_pairwise_relationship_with_logprobs_flexible(
    "Exercise",
    "Weight Loss",
    openai_client=azure_openai_client,
    log_probs=True,  # Enable log probabilities
    confidence_level=True  # Enable confidence level
)

print("Answer:", result_with_logprobs.get("answer"))
print("Has logprobs data:", "logprobs" in result_with_logprobs)
if result_with_logprobs.get("answer_choice_logprobs"):
    print("A/B/C logprobs:", json.dumps(result_with_logprobs.get("answer_choice_logprobs", {}), indent=2))

print("\n⚡ Testing with log_probs=False (faster, basic analysis):")
print("=" * 60)

result_without_logprobs = modeler.suggest_pairwise_relationship_with_logprobs_flexible(
    "Exercise",
    "Weight Loss", 
    openai_client=azure_openai_client,
    log_probs=False,  # Disable log probabilities for faster execution
    confidence_level=True  # Enable confidence level
)

print("Answer:", result_without_logprobs.get("answer"))
print("Has logprobs data:", "logprobs" in result_without_logprobs)
print("Keys in result:", list(result_without_logprobs.keys()))

print("\n📊 Performance comparison:")
print(f"With logprobs - Keys: {len(result_with_logprobs.keys())}")
print(f"Without logprobs - Keys: {len(result_without_logprobs.keys())}")
print("\n✅ Both results have the same answer structure but different data richness!")

Answer: A
Has logprobs data: True

⚡ Testing with log_probs=False (faster, basic analysis):
Answer: A
Has logprobs data: False
Keys in result: ['result', 'description', 'answer', 'confidence']

📊 Performance comparison:
With logprobs - Keys: 8
Without logprobs - Keys: 4

✅ Both results have the same answer structure but different data richness!
Answer: A
Has logprobs data: False
Keys in result: ['result', 'description', 'answer', 'confidence']

📊 Performance comparison:
With logprobs - Keys: 8
Without logprobs - Keys: 4

✅ Both results have the same answer structure but different data richness!


In [12]:
result_with_logprobs

{'result': ['Exercise',
  'Weight Loss',
  'The more likely cause-and-effect relationship is A. Exercise causes Weight Loss. Regular physical activity increases energy expenditure, which can lead to a calorie deficit and subsequent weight loss if not offset by increased caloric intake. While weight loss might motivate some people to exercise more, it is not a direct cause in the same way. Therefore, the most direct and commonly supported causal relationship is that exercise leads to weight loss.\n\n<answer>A</answer>'],
 'description': 'The more likely cause-and-effect relationship is A. Exercise causes Weight Loss. Regular physical activity increases energy expenditure, which can lead to a calorie deficit and subsequent weight loss if not offset by increased caloric intake. While weight loss might motivate some people to exercise more, it is not a direct cause in the same way. Therefore, the most direct and commonly supported causal relationship is that exercise leads to weight loss.\

In [13]:
result_without_logprobs

{'result': ['Exercise',
  'Weight Loss',
  'Exercise is a physical activity that increases energy expenditure, which can lead to burning calories and, over time, weight loss if not offset by increased calorie intake. Therefore, it is reasonable to conclude that exercise can cause weight loss. While losing weight might make exercise easier or more enjoyable, it does not directly cause someone to exercise. Thus, the most likely cause-and-effect relationship is:\n\n<answer>A</answer>'],
 'description': 'Exercise is a physical activity that increases energy expenditure, which can lead to burning calories and, over time, weight loss if not offset by increased calorie intake. Therefore, it is reasonable to conclude that exercise can cause weight loss. While losing weight might make exercise easier or more enjoyable, it does not directly cause someone to exercise. Thus, the most likely cause-and-effect relationship is:\n\n<answer>A</answer>',
 'answer': 'A'}

This is the function used, added to Simple model Suggester

In [ ]:
import re
import math
from openai import OpenAI

def suggest_pairwise_relationship_with_logprobs_flexible(self, variable1: str, variable2: str, openai_client=None, model_name="gpt-4o-mini", log_probs=True, confidence_level=True):
    """
    Suggests a cause-and-effect relationship with optional detailed log probabilities.
    Uses OpenAI client directly for full logprob access when enabled.
    
    Args:
        variable1 (str): The name of the first variable.
        variable2 (str): The name of the second variable.
        openai_client: Optional OpenAI client instance. If None, uses guidance model.
        model_name (str): Model name to use with OpenAI client.
        log_probs (bool): Whether to calculate log probabilities. Default True.
        confidence_level (bool): Whether to request confidence score. Default False.
        
    Returns:
        dict: Contains 'result', 'description', 'answer', optionally 'logprobs' data, and optionally 'confidence'.
    """
    # If no OpenAI client provided, try to create one from environment
    if openai_client is None:
        import os
        # Try to extract connection details from guidance model
        if hasattr(self.llm, 'engine'):
            try:
                openai_client = OpenAI(
                    api_key=os.environ.get("OPENAI_API_KEY"),
                    base_url=os.environ.get("OPENAI_BASE_URL")
                )
            except:
                raise ValueError("Could not create OpenAI client. Please pass openai_client parameter.")
    
    # Confidence instruction
    confidence_instruction = ""
    if confidence_level:
        confidence_instruction = " Also provide a confidence score between 0 and 1 within <confidence></confidence> tags, regardless of whether the answer is A, B, or C."
    
    # Construct the prompt
    prompt = f"""Which cause-and-effect-relationship is more likely? Provide reasoning and give your final answer (A, B, or C) in <answer> </answer> tags with the letter only and no whitespaces.{confidence_instruction}
A. {variable1} causes {variable2} 
B. {variable2} causes {variable1} 
C. neither {variable1} nor {variable2} cause each other."""
    
    # Make API call with conditional logprobs
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a helpful assistant for causal reasoning."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        logprobs=log_probs,  # Use the boolean parameter
        top_logprobs=3 if log_probs else None,  # Only request top logprobs if needed
        max_tokens=200
    )
    
    choice = response.choices[0]
    description = choice.message.content
    
    # Extract answer
    answer = re.findall(r"<answer>(.*?)</answer>", description)
    answer = [ans.strip() for ans in answer]
    answer_str = "".join(answer) if answer else ""

    # Extract confidence if requested
    confidence_score = None
    if confidence_level:
        confidence_match = re.findall(r"<confidence>(.*?)</confidence>", description)
        if confidence_match:
            try:
                confidence_score = float(confidence_match[0].strip())
            except ValueError:
                confidence_score = None

    # Determine result based on the chosen answer
    if answer_str == "A":
        result = [variable1, variable2, description]
    elif answer_str == "B":
        result = [variable2, variable1, description]
    elif answer_str == "C":
        result = [None, None, description]
    else:
        result = [None, None, description]

    # Base return dictionary
    return_dict = {
        'result': result,
        'description': description,
        'answer': answer_str,
    }
    
    # Add confidence score if requested
    if confidence_level:
        return_dict['confidence'] = confidence_score
    
    # Only process logprobs if requested
    if log_probs:
        logprobs_data = []
        if getattr(choice, "logprobs", None) and getattr(choice.logprobs, "content", None):
            logprobs_data = choice.logprobs.content

        answer_token_logprob = None
        answer_choice_logprobs = {}
        answer_tokens_logprobs = []

        if logprobs_data:
            reconstructed_text = ""
            token_spans = []
            for token_item in logprobs_data:
                token_text = getattr(token_item, "token", "") or ""
                start_idx = len(reconstructed_text)
                reconstructed_text += token_text
                end_idx = len(reconstructed_text)
                token_spans.append((start_idx, end_idx, token_item))

            match = re.search(r"<answer>(.*?)</answer>", reconstructed_text, flags=re.DOTALL)
            if match:
                content_start, content_end = match.span(1)
                for start_idx, end_idx, token_item in token_spans:
                    overlaps_answer = start_idx < content_end and end_idx > content_start
                    if overlaps_answer:
                        token_text = getattr(token_item, "token", "") or ""
                        token_logprob = getattr(token_item, "logprob", None)
                        answer_tokens_logprobs.append({
                            "token": token_text,
                            "logprob": token_logprob
                        })

                        token_letter = token_text.strip()
                        if token_letter in {"A", "B", "C"} and token_logprob is not None:
                            answer_choice_logprobs[token_letter] = token_logprob

                        top_alternatives = getattr(token_item, "top_logprobs", None) or []
                        for alt in top_alternatives:
                            alt_token = (getattr(alt, "token", "") or "").strip()
                            if alt_token in {"A", "B", "C"}:
                                alt_logprob = getattr(alt, "logprob", None)
                                if alt_logprob is not None:
                                    answer_choice_logprobs[alt_token] = alt_logprob
                    if start_idx <= content_start and end_idx > content_start:
                        if answer_token_logprob is None:
                            answer_token_logprob = getattr(token_item, "logprob", None)

        # Add logprobs data to return dictionary
        return_dict.update({
            'logprobs': logprobs_data,
            'answer_token_logprob': answer_token_logprob,
            'answer_choice_logprobs': answer_choice_logprobs,
            'answer_tokens_logprobs': answer_tokens_logprobs,
        })

    return return_dict

In [ ]:
# Test the SUPER SIMPLIFIED function
from types import MethodType

# Add the super simple method to the modeler
modeler.suggest_pairwise_relationship_with_logprobs_super_simple = MethodType(
    suggest_pairwise_relationship_with_logprobs_super_simple, modeler
)

print("🚀🚀 Testing SUPER SIMPLIFIED function:")
print("=" * 60)

result_super_simple = modeler.suggest_pairwise_relationship_with_logprobs_super_simple(
    "Exercise",
    "Weight Loss",
    openai_client=azure_openai_client,
    log_probs=True,
    confidence_level=True,
    debug=True  # Enable debug output
)

print("\n📊 Results:")
print(f"Answer: {result_super_simple.get('answer')}")
print(f"Answer token text: {result_super_simple.get('answer_token_text')}")
print(f"Answer token logprob: {result_super_simple.get('answer_token_logprob')}")
print(f"Answer choice logprobs: {result_super_simple.get('answer_choice_logprobs')}")

# Check if it worked
answer_logprob = result_super_simple.get('answer_token_logprob')
if answer_logprob is None:
    print(f"\n❌ SUPER SIMPLIFIED version: Answer token logprob is EMPTY!")
else:
    print(f"\n✅ SUPER SIMPLIFIED version: Answer token logprob is NOT empty! Value: {answer_logprob}")

print(f"\n🎯 This approach is based on your insight: extract answer first, then find that exact token!")
print("🧠 Much more direct than complex text reconstruction approaches.")

🚀🚀 Testing SUPER SIMPLIFIED function:
🎯 DEBUG: Extracted answer from text: 'A'
🔍 DEBUG: Searching for token 'A' in 102 tokens
🔍 DEBUG: Token 'Exercise' -> 'Exercise' != 'A'
🔍 DEBUG: Token ' is' -> 'is' != 'A'
🔍 DEBUG: Token ' a' -> 'a' != 'A'
🔍 DEBUG: Token ' physical' -> 'physical' != 'A'
🔍 DEBUG: Token ' activity' -> 'activity' != 'A'
🔍 DEBUG: Token ' that' -> 'that' != 'A'
🔍 DEBUG: Token ' increases' -> 'increases' != 'A'
🔍 DEBUG: Token ' energy' -> 'energy' != 'A'
🔍 DEBUG: Token ' expenditure' -> 'expenditure' != 'A'
🔍 DEBUG: Token ',' -> ',' != 'A'
🔍 DEBUG: Token ' which' -> 'which' != 'A'
🔍 DEBUG: Token ' can' -> 'can' != 'A'
🔍 DEBUG: Token ' lead' -> 'lead' != 'A'
🔍 DEBUG: Token ' to' -> 'to' != 'A'
🔍 DEBUG: Token ' a' -> 'a' != 'A'
🔍 DEBUG: Token ' calorie' -> 'calorie' != 'A'
🔍 DEBUG: Token ' deficit' -> 'deficit' != 'A'
🔍 DEBUG: Token ' and' -> 'and' != 'A'
🔍 DEBUG: Token ',' -> ',' != 'A'
🔍 DEBUG: Token ' over' -> 'over' != 'A'
🔍 DEBUG: Token ' time' -> 'time' != 'A'
🔍 DEBUG

In [12]:
result_with_logprobs

{'result': ['Exercise',
  'Weight Loss',
  'Exercise is a physical activity that increases energy expenditure, which can lead to a calorie deficit and, over time, weight loss. While losing weight may motivate some people to exercise more, the direct causal relationship is stronger from exercise to weight loss. There is substantial scientific evidence supporting that regular exercise contributes to weight loss, whereas weight loss itself does not inherently cause someone to exercise. Therefore, the most likely cause-and-effect relationship is A.\n\n<answer>A</answer>\n<confidence>0.95</confidence>'],
 'description': 'Exercise is a physical activity that increases energy expenditure, which can lead to a calorie deficit and, over time, weight loss. While losing weight may motivate some people to exercise more, the direct causal relationship is stronger from exercise to weight loss. There is substantial scientific evidence supporting that regular exercise contributes to weight loss, wherea

In [19]:
import re
import math
from openai import OpenAI

def suggest_pairwise_relationship_with_logprobs_flexible(self, variable1: str, variable2: str, openai_client=None, model_name="gpt-4o-mini", log_probs=True, confidence_level=True):
    """
    Suggests a cause-and-effect relationship with optional detailed log probabilities.
    Uses OpenAI client directly for full logprob access when enabled.
    
    Args:
        variable1 (str): The name of the first variable.
        variable2 (str): The name of the second variable.
        openai_client: Optional OpenAI client instance. If None, uses guidance model.
        model_name (str): Model name to use with OpenAI client.
        log_probs (bool): Whether to calculate log probabilities. Default True.
        confidence_level (bool): Whether to request confidence and strength scores. Default True.
        
    Returns:
        dict: Contains 'result', 'description', 'answer', optionally 'logprobs' data, 
              and optionally 'confidence' and 'strength' scores.
    """
    # If no OpenAI client provided, try to create one from environment
    if openai_client is None:
        import os
        # Try to extract connection details from guidance model
        if hasattr(self.llm, 'engine'):
            try:
                openai_client = OpenAI(
                    api_key=os.environ.get("OPENAI_API_KEY"),
                    base_url=os.environ.get("OPENAI_BASE_URL")
                )
            except:
                raise ValueError("Could not create OpenAI client. Please pass openai_client parameter.")
    
    # Confidence and strength instruction
    confidence_instruction = ""
    if confidence_level:
        confidence_instruction = """ 
Additionally, provide TWO scores within tags:
1. <confidence></confidence>: Your confidence in this causal judgment (0-1 scale)
   - How certain are you that you chose the correct relationship (A, B, or C)?
   - Example: High confidence = 0.9 (very sure), Low confidence = 0.5 (uncertain)

2. <strength></strength>: The strength of the causal relationship (0-1 scale)
   - IF a causal relationship exists (A or B), how strong/deterministic is it?
   - 1.0 = Deterministic relationship (e.g., "Decapitation → Death")
   - 0.85 = Very strong relationship (e.g., "Smoking → Lung Cancer")
   - 0.7 = Moderate relationship (e.g., "Age → Heart Attack" - depends on genetics, lifestyle)
   - 0.5 = Moderate-weak relationship (e.g., "Education → Income" - many exceptions)
   - 0.3 = Weak relationship (e.g., "Birth Order → Personality" - small effect, many confounders)
   - 0.1 = Very weak relationship (e.g., "Moon Phase → Mood" - questionable evidence)
   - 0.0 = No causal relationship (only use if answer is C)
   
Important distinctions:
- You can have HIGH confidence (0.9) that a WEAK relationship (0.3) exists
- You can have LOW confidence (0.5) about a potentially STRONG relationship (0.8)
- If answer is C (no relationship), set strength to 0.0 but confidence reflects certainty of no relationship
- Strength reflects the SIZE of the causal effect, not your confidence in the judgment
"""
    
    # Construct the prompt
    prompt = f"""Which cause-and-effect-relationship is more likely? Provide reasoning and give your final answer (A, B, or C) in <answer> </answer> tags with the letter only and no whitespaces.{confidence_instruction}
A. {variable1} causes {variable2} 
B. {variable2} causes {variable1} 
C. neither {variable1} nor {variable2} cause each other."""
    
    # Make API call with conditional logprobs
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": "You are a helpful assistant for causal reasoning."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
        logprobs=log_probs,  # Use the boolean parameter
        top_logprobs=3 if log_probs else None,  # Only request top logprobs if needed
        max_tokens=300  # Increased to accommodate strength explanation
    )
    
    choice = response.choices[0]
    description = choice.message.content
    
    # Extract answer
    answer = re.findall(r"<answer>(.*?)</answer>", description)
    answer = [ans.strip() for ans in answer]
    answer_str = "".join(answer) if answer else ""

    # Extract confidence and strength if requested
    confidence_score = None
    strength_score = None
    
    if confidence_level:
        # Extract confidence
        confidence_match = re.findall(r"<confidence>(.*?)</confidence>", description)
        if confidence_match:
            try:
                confidence_score = float(confidence_match[0].strip())
                confidence_score = max(0.0, min(1.0, confidence_score))  # Clamp to [0, 1]
            except ValueError:
                confidence_score = None
        
        # Extract strength
        strength_match = re.findall(r"<strength>(.*?)</strength>", description)
        if strength_match:
            try:
                strength_score = float(strength_match[0].strip())
                strength_score = max(0.0, min(1.0, strength_score))  # Clamp to [0, 1]
            except ValueError:
                strength_score = None

    # Determine result based on the chosen answer
    if answer_str == "A":
        result = [variable1, variable2, description]
    elif answer_str == "B":
        result = [variable2, variable1, description]
    elif answer_str == "C":
        result = [None, None, description]
    else:
        result = [None, None, description]

    # Base return dictionary
    return_dict = {
        'result': result,
        'description': description,
        'answer': answer_str,
    }
    
    # Add confidence and strength scores if requested
    if confidence_level:
        return_dict['confidence_score'] = confidence_score
        return_dict['strength_score'] = strength_score
    
    # Only process logprobs if requested
    if log_probs:
        logprobs_data = []
        if getattr(choice, "logprobs", None) and getattr(choice.logprobs, "content", None):
            logprobs_data = choice.logprobs.content

        answer_token_logprob = None
        answer_choice_logprobs = {}
        answer_tokens_logprobs = []

        if logprobs_data:
            reconstructed_text = ""
            token_spans = []
            for token_item in logprobs_data:
                token_text = getattr(token_item, "token", "") or ""
                start_idx = len(reconstructed_text)
                reconstructed_text += token_text
                end_idx = len(reconstructed_text)
                token_spans.append((start_idx, end_idx, token_item))

            match = re.search(r"<answer>(.*?)</answer>", reconstructed_text, flags=re.DOTALL)
            if match:
                content_start, content_end = match.span(1)
                for start_idx, end_idx, token_item in token_spans:
                    overlaps_answer = start_idx < content_end and end_idx > content_start
                    if overlaps_answer:
                        token_text = getattr(token_item, "token", "") or ""
                        token_logprob = getattr(token_item, "logprob", None)
                        answer_tokens_logprobs.append({
                            "token": token_text,
                            "logprob": token_logprob
                        })

                        # FIXED: Clean token properly to handle ">A", "<answer", etc.
                        token_clean = ''.join(c for c in token_text if c.isalnum())
                        if token_clean in {"A", "B", "C"} and token_logprob is not None:
                            answer_choice_logprobs[token_clean] = token_logprob

                        top_alternatives = getattr(token_item, "top_logprobs", None) or []
                        for alt in top_alternatives:
                            alt_token = getattr(alt, "token", "") or ""
                            alt_clean = ''.join(c for c in alt_token if c.isalnum())
                            if alt_clean in {"A", "B", "C"}:
                                alt_logprob = getattr(alt, "logprob", None)
                                if alt_logprob is not None:
                                    answer_choice_logprobs[alt_clean] = alt_logprob
                    
                    if start_idx <= content_start and end_idx > content_start:
                        if answer_token_logprob is None:
                            answer_token_logprob = getattr(token_item, "logprob", None)

        # Add logprobs data to return dictionary
        return_dict.update({
            'logprobs': logprobs_data,
            'answer_token_logprob': answer_token_logprob,
            'answer_choice_logprobs': answer_choice_logprobs,
            'answer_tokens_logprobs': answer_tokens_logprobs,
        })

    return return_dict

In [20]:
print("=" * 60)
modeler.suggest_pairwise_relationship_with_logprobs_flexible = MethodType(
    suggest_pairwise_relationship_with_logprobs_flexible, modeler
)

result_super_simple = modeler.suggest_pairwise_relationship_with_logprobs_flexible(
    "Exercise",
    "Weight Loss",
    openai_client=azure_openai_client,
    log_probs=True,
    confidence_level=True,
   
)

In [21]:
result_super_simple

{'result': ['Exercise',
  'Weight Loss',
  '<answer>A</answer>\n<confidence>0.95</confidence>\n<strength>0.8</strength>\n\nReasoning:\nExercise is a well-established cause of weight loss, primarily through increased energy expenditure and metabolic changes. While weight loss may sometimes motivate people to exercise more, the direct causal pathway from exercise to weight loss is much stronger and better supported by scientific evidence. There is no strong evidence that weight loss itself directly causes exercise, and the possibility that neither causes the other is highly unlikely given the extensive research on exercise and weight management.'],
 'description': '<answer>A</answer>\n<confidence>0.95</confidence>\n<strength>0.8</strength>\n\nReasoning:\nExercise is a well-established cause of weight loss, primarily through increased energy expenditure and metabolic changes. While weight loss may sometimes motivate people to exercise more, the direct causal pathway from exercise to weigh